In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)
import sys
# sys.path.append(r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data processing\functions")
sys.path.append(r"/ihome/ylee/yiz133/Code/Data processing/functions/")
import mdata_utils 


In [4]:
path = "/ix1/ylee/Yifan_Zhang/Code_data/Tumor/GSE139555_2019/data/Processed/"
mdata = mu.read(path + "CD8_DE2500_singleVDJ_emb.h5mu")

In [ ]:
mdata.obs['subtype'].value_counts()

In [ ]:
mdata = mdata[mdata.obs['subtype'].isin(['CD8_Tem', 'CD8_Teff', 'CD8_Trm_exh_L', 
                              'CD8_Trm_exh_H','CD8_Trm_exh_M'])].copy()
mdata

In [ ]:
mdata.obs['subtype'].value_counts()

In [ ]:
mdata.obs['patient'].value_counts()

In [ ]:
pd.DataFrame({
    "CDR3b": mdata.obs["VDJ_1_cdr3_aa"].astype(str),
    "Vb": mdata.obs["VDJ_1_v_call"].astype(str),
    "Jb": mdata.obs["VDJ_1_j_call"].astype(str),
    "CDR3a": mdata.obs["VJ_1_cdr3_aa"].astype(str),
    "condition": mdata.obs["patient"].astype(str),
    "clone_id_size": mdata["airr"].obs["clone_id_size"],
}).to_csv("CD8_singleVDJ_tcr.csv", index=False)

## Clonotype trace

In [ ]:
ir.pp.ir_dist(mdata,metric="tcrdist",sequence="aa",cutoff=15)
ir.tl.define_clonotype_clusters(mdata, sequence="aa", metric="tcrdist", receptor_arms="all", dual_ir="any")

In [ ]:
obs = mdata.obs.copy()

n_alpha = obs['VJ_1_cdr3_aa'].nunique()
n_beta  = obs['VDJ_1_cdr3_aa'].nunique()

print(f"Unique alpha-chain (VJ) clonotypes:  {n_alpha}")
print(f"Unique beta-chain  (VDJ) clonotypes: {n_beta}")
print(f"Total cells: {len(obs)}")

In [ ]:
# Subset to clonotypes appearing in multiple source tissues AND multiple subtypes
clone_col = 'cc_aa_tcrdist' if 'cc_aa_tcrdist' in mdata['airr'].obs.columns else 'clone_id'

obs_all = mdata.obs.copy()
obs_all['_clone'] = mdata['airr'].obs[clone_col].values
obs_all['_source'] = mdata['gex'].obs['source'].values
obs_all['_subtype'] = mdata['gex'].obs['subtype'].values

obs_valid = obs_all.dropna(subset=['_clone', '_source', '_subtype'])

source_nunique = obs_valid.groupby('_clone')['_source'].nunique()
subtype_nunique = obs_valid.groupby('_clone')['_subtype'].nunique()

multi_source = set(source_nunique[source_nunique >= 3].index)
print(f"Clonotypes in 2+ sources:  {len(multi_source)}")

mask = obs_valid['_clone'].isin(multi_source)
mdata_multi = mdata[mask.values].copy()
# Plot clonotype network colored by source and subtype
ir.pp.ir_dist(mdata_multi,metric="tcrdist",sequence="aa",cutoff=15)
ir.tl.define_clonotype_clusters(mdata_multi, sequence="aa", metric="tcrdist", receptor_arms="all", dual_ir="any")

ir.tl.clonotype_network(mdata_multi, min_cells=2, sequence="aa", metric="tcrdist")
_ = ir.pl.clonotype_network(mdata_multi, color="gex:source",
      label_fontsize=5, panel_size=(7, 7), base_size=10)



In [ ]:
multi_subtype = set(subtype_nunique[subtype_nunique >= 3].index)
print(f"Clonotypes in 2+ subtypes: {len(multi_subtype)}")
mask_type = obs_valid['_clone'].isin(multi_subtype)
mdata_multi = mdata[mask_type.values].copy()
# Plot clonotype network colored by source and subtype
ir.pp.ir_dist(mdata_multi,metric="tcrdist",sequence="aa",cutoff=15)
ir.tl.define_clonotype_clusters(mdata_multi, sequence="aa", metric="tcrdist", receptor_arms="all", dual_ir="any")

ir.tl.clonotype_network(mdata_multi, min_cells=4, sequence="aa", metric="tcrdist")
_ = ir.pl.clonotype_network(mdata_multi, color="gex:subtype",
      label_fontsize=5, panel_size=(7,13), base_size=10)


multi_both = multi_source & multi_subtype

In [ ]:
# Exhaustion gene-set score + per-cell slope across ordered genes (obs: "ex trend")
from scipy import sparse
from scipy.stats import linregress

# Use full mdata GEX so all cells get scores; copy onto the clonotype_network subset (mdata_multi)
gex = mdata["gex"]
# Order: CD38, CD39, PD-1, CTLA4, TIGIT, LAG3, TOX, HAVCR2 — use symbols present in var_names
_candidates = [
    ("CD38", ["CD38"]),
    ("CD39", ["ENTPD1", "CD39"]),
    ("PD-1", ["PDCD1"]),
    ("CTLA4", ["CTLA4"]),
    ("TIGIT", ["TIGIT"]),
    ("LAG3", ["LAG3"]),
    ("TOX", ["TOX"]),
    ("HAVCR2", ["HAVCR2"]),
]
genes_present = []
for _label, syms in _candidates:
    for s in syms:
        if s in gex.var_names:
            genes_present.append(s)
            break

print("Exhaustion genes used (in order):", genes_present)
if len(genes_present) == 0:
    raise ValueError("No exhaustion genes found in gex.var_names")

sc.tl.score_genes(gex, gene_list=genes_present, score_name="exhaustion_score")

Xsub = gex[:, genes_present].X
if sparse.issparse(Xsub):
    Xsub = Xsub.toarray()
n_genes = Xsub.shape[1]
if n_genes < 2:
    slopes = np.full(Xsub.shape[0], np.nan, dtype=float)
else:
    xi = np.arange(n_genes, dtype=float)
    slopes = np.array([linregress(xi, Xsub[i]).slope for i in range(Xsub.shape[0])], dtype=float)

gex.obs["ex trend"] = slopes
mdata.obs["ex trend"] = gex.obs["ex trend"]
mdata.obs["exhaustion_score"] = gex.obs["exhaustion_score"]
# Clonotype_network subset (mdata_multi): align by cell name
_sub = gex.obs.reindex(mdata_multi.obs_names)
mdata_multi.obs["ex trend"] = _sub["ex trend"].values
mdata_multi.obs["exhaustion_score"] = _sub["exhaustion_score"].values
mdata_multi["gex"].obs["ex trend"] = _sub["ex trend"].values
mdata_multi["gex"].obs["exhaustion_score"] = _sub["exhaustion_score"].values

# DataFrame: TCR embedding columns + ex trend (one row per cell, index = obs_names)
tcr_X = np.asarray(mdata.obsm["tcr_embs"])
feat_names = list(mdata.uns.get("tcr_embs_feature_names", []))
if len(feat_names) != tcr_X.shape[1]:
    feat_names = [f"tcr_emb_{i}" for i in range(tcr_X.shape[1])]
df_tcr_ex = pd.DataFrame(tcr_X, index=mdata.obs_names, columns=feat_names)
df_tcr_ex["ex trend"] = mdata.obs["ex trend"].values

# Aggregate to one row per clone: TCR emb (first cell, identical within clone) + mean ex trend
_clone_col = 'cc_aa_tcrdist' if 'cc_aa_tcrdist' in mdata['airr'].obs.columns else 'clone_id'
df_tcr_ex["clone_id"] = mdata["airr"].obs[_clone_col].values
df_tcr_ex_valid = df_tcr_ex.dropna(subset=["clone_id"])
df_tcr_ex_clone = (
    df_tcr_ex_valid
    .groupby("clone_id")
    .agg({**{c: "first" for c in feat_names}, "ex trend": "mean"})
)
df_tcr_ex_clone.index.name = "clone_id"
print(f"Clone-level df: {df_tcr_ex_clone.shape[0]} clones × {df_tcr_ex_clone.shape[1]} cols")


In [ ]:
df_tcr_ex_clone.to_csv('tcr_ex_clone.csv')

## DEGs

In [ ]:
mdata

In [ ]:
mdata.obs['source'].value_counts()

In [ ]:
import os, re
from scipy import stats
from scipy.sparse import issparse

corr_method = 'pearson'  # choose: 'pearson' or 'spearman'
corr_method = str(corr_method).strip().lower()
if corr_method not in {'pearson', 'spearman'}:
    raise ValueError("corr_method must be 'pearson' or 'spearman'")

# Gate Fisher test: both |r| above this and both per-group correlation p-values below fisher_max_p_individual
fisher_min_abs_r = 0.1
fisher_max_p_individual = 0.1


def _corr_r(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    if corr_method == 'spearman':
        r, p = stats.spearmanr(x, y)
    else:
        r, p = stats.pearsonr(x, y)
    return r, p


def _bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    q = np.empty(m)
    running = 1.0
    for k in range(m - 1, -1, -1):
        idx = order[k]
        q[idx] = min(running, pvals[idx] * m / (k + 1))
        running = q[idx]
    return np.clip(q, 0.0, 1.0)


def _fisher_z_corr_diff_p(r1, n1, r2, n2):
    if n1 < 4 or n2 < 4:
        return np.nan
    if not (np.isfinite(r1) and np.isfinite(r2)):
        return np.nan
    z1 = np.arctanh(np.clip(r1, -0.999999, 0.999999))
    z2 = np.arctanh(np.clip(r2, -0.999999, 0.999999))
    se = np.sqrt(1.0 / (n1 - 3) + 1.0 / (n2 - 3))
    z_stat = (z1 - z2) / se
    return 2.0 * (1.0 - stats.norm.cdf(abs(z_stat)))



In [ ]:
mdata.obs['subtype'].value_counts()

In [ ]:
top_n = 30
file_name = 'Tumor_s_vs_multi'

outdir = file_name
target_groups = ['Tumor_multiSite_clone', 'Tumor_singleSite_clone']
os.makedirs(outdir, exist_ok=True)

# Loop through each subtype and compare:
#   group1: in_two_tissue == True
#   group2: cloned == True AND in_two_tissue == False

subtypes = mdata.obs['subtype'].astype(str).unique().tolist()
df_list = []
corr_sig_list = []

for st in subtypes:
    m_sub = mdata[(mdata.obs['subtype'].astype(str) == st)].copy()
    m_sub = m_sub[m_sub.obs['clone_status'].isin(target_groups)].copy()
    if m_sub.obs.shape[0] <= 10:
        continue

    sc.tl.rank_genes_groups(m_sub['gex'], 
    groups= target_groups,
    groupby='clone_status')

    # Save DEG table for this subtype
    for g in target_groups:
        dfi = sc.get.rank_genes_groups_df(m_sub['gex'], group=g)
        dfi['subtype'] = st
        dfi['deg_group'] = g
        dfi = dfi[(dfi['pvals_adj'] < 0.05) & (dfi['logfoldchanges'].abs() > 0.5)]

        df_list.append(dfi.head(top_n))

        genes_top = dfi['names'].head(top_n).tolist()
        gex = m_sub['gex']
        tcr = m_sub.obsm['tcr_embs']
        cs = m_sub.obs['clone_status'].astype(str).to_numpy()
        if len(target_groups) >= 2:
            g1, g2 = target_groups[0], target_groups[1]
            m1 = cs == g1
            m2 = cs == g2
            n1, n2 = int(m1.sum()), int(m2.sum())
            if n1 >= 4 and n2 >= 4:
                mtx = gex[:, genes_top].X
                if issparse(mtx):
                    mtx = mtx.toarray()
                mtx = np.asarray(mtx)
                if mtx.ndim == 1:
                    mtx = mtx.reshape(-1, 1)
                for gi, gene in enumerate(genes_top):
                    y = np.asarray(mtx[:, gi]).ravel()
                    for j in range(tcr.shape[1]):
                        yi1, vi1 = y[m1], tcr[m1, j]
                        yi2, vi2 = y[m2], tcr[m2, j]
                        r1, p1 = _corr_r(yi1, vi1)
                        r2, p2 = _corr_r(yi2, vi2)
                        r_all, p_all = _corr_r(y, tcr[:, j])
                        if (
                            np.isfinite(r1)
                            and np.isfinite(r2)
                            and np.isfinite(p1)
                            and np.isfinite(p2)
                            and abs(r1) > fisher_min_abs_r
                            and abs(r2) > fisher_min_abs_r
                            and p1 < fisher_max_p_individual
                            and p2 < fisher_max_p_individual
                        ):
                            p_diff = _fisher_z_corr_diff_p(r1, n1, r2, n2)
                        else:
                            p_diff = np.nan
                        corr_sig_list.append(
                            {
                                'subtype': st,
                                'deg_group': g,
                                'gene': gene,
                                'tcr_col': j,
                                'r_pooled': r_all,
                                'p_pooled': p_all,
                                'r_' + g1: r1,
                                'r_' + g2: r2,
                                'p_' + g1: p1,
                                'p_' + g2: p2,
                                'p_diff': p_diff,
                                'n_' + g1: n1,
                                'n_' + g2: n2,
                            }
                        )

    # sc.pl.rank_genes_groups_dotplot(
    #     m_sub['gex'],
    #     groups= target_groups,
    #     groupby='clone_status',
    #     n_genes=top_n,
    #     standard_scale='var',
    #     show=False
    # )

    # st_safe = re.sub(r"[^A-Za-z0-9_\-]+", "_", st)
    # plt.savefig(
    #     os.path.join(outdir, f"deg_dotplot_{st_safe}.png"),
    #     dpi=300,
    #     bbox_inches='tight'
    # )
    # plt.close()


In [ ]:
len(df_list)

In [ ]:
# Combined DEG results across all subtypes and both groups
df = pd.concat(df_list, ignore_index=True) if len(df_list) > 0 else pd.DataFrame()

corr_df_all = (
    pd.DataFrame(corr_sig_list) if corr_sig_list else pd.DataFrame()
)
tcr_feature_names = mdata.uns.get('tcr_embs_feature_names', [])

def _tcr_idx_to_name(idx):
    idx = int(idx)
    if 0 <= idx < len(tcr_feature_names):
        return tcr_feature_names[idx]
    return f'tcr_col_{idx}'
if not corr_df_all.empty:
    corr_df_all['p_adj'] = _bh_fdr(corr_df_all['p_diff'].fillna(1.0).to_numpy())
    corr_df_all['tcr_col_name'] = corr_df_all['tcr_col'].map(_tcr_idx_to_name)

    #  Filter in the DEG list
    corr_df_sig = corr_df_all[(corr_df_all['p_adj'] < 0.05)].copy()
    corr_df_sig['tcr_col_idx'] = corr_df_sig['tcr_col']
    corr_df_sig['tcr_col'] = corr_df_sig['tcr_col_name']
else:
    corr_df_sig = corr_df_all

df.head()
corr_df_sig.head()


In [ ]:
df.to_csv(os.path.join(outdir, 'DEG_dual_single_clones.csv'))
corr_df_sig.to_csv(os.path.join(outdir, 'gene_tcr_corr_sig_fdr05.csv'))
corr_df_all.to_csv(os.path.join(outdir, 'gene_tcr_corr_all_tests.csv'))

In [ ]:
mdata.obs['clone_status'].value_counts()

In [ ]:
sc.pl.umap(mdata['gex'], color=['clone_status', 'subtype','source'], wspace=0.5, ncols=2)

In [ ]:
# Top K common (gene, tcr_col) pairs across subtypes; ranked by subtype_count
if corr_df_sig.empty:
    top10_common_pairs = pd.DataFrame(
        columns=['gene', 'tcr_col_idx', 'tcr_col', 'subtype_count', 'row_count']
    )
else:
    subtype_total = corr_df_sig['subtype'].nunique()
    pair_common = (
        corr_df_sig.groupby(['gene', 'tcr_col_idx'], as_index=False)
        .agg(
            tcr_col=('tcr_col', 'first'),
            subtype_count=('subtype', 'nunique'),
            row_count=('gene', 'size'),
        )
        .sort_values(
            ['subtype_count', 'row_count', 'gene', 'tcr_col_idx'],
            ascending=[False, False, True, True],
        )
    )
    # Pairs significant in at least 2 subtypes; top 10 by subtype_count
    top10_common_pairs = pair_common[pair_common['subtype_count'] >= 2].head(top_n)
    print(f"Total subtypes represented in corr_df_sig: {subtype_total}")

if top10_common_pairs.empty:
    print('No common (gene, tcr_col) pairs across multiple subtypes in corr_df_sig.')
else:
    top10_common_pairs

In [ ]:
pair_common_subtype = pair_common[pair_common['subtype_count'] > 1]
pair_common_subtype

In [ ]:
len(set(pair_common_subtype['gene'].unique()))

In [ ]:
# First row of pair_common_subtype: one panel per subtype; target_groups as colors; dashed lines = per-group OLS (r from corr_df_sig)
from scipy.sparse import issparse
from scipy import stats

row_idx = 1
scatter_size = 5

if pair_common_subtype.empty:
    print('pair_common_subtype is empty; run the pair_common cell first.')
else:
    r = pair_common_subtype.iloc[row_idx]
    gene = r['gene']
    j = int(r['tcr_col_idx'])
    tname = r['tcr_col']

    tg = target_groups if 'target_groups' in globals() else [
        'Tumor_multiSite_clone',
        'Tumor_singleSite_clone',
    ]
    corr_method = globals().get('corr_method', 'pearson')
    corr_method = str(corr_method).strip().lower()
    if corr_method not in {'pearson', 'spearman', 'mutual_info'}:
        raise ValueError("corr_method must be 'pearson', 'spearman', or 'mutual_info'")
    if len(tg) < 2:
        raise ValueError('Need at least two target_groups for a two-color plot.')
    g1, g2 = tg[0], tg[1]

    if corr_df_sig.empty or 'tcr_col_idx' not in corr_df_sig.columns:
        print('corr_df_sig missing or has no tcr_col_idx; run the correlation cell first.')
    else:
        cd = (
            corr_df_sig[
                (corr_df_sig['gene'] == gene) & (corr_df_sig['tcr_col_idx'] == j)
            ]
            .drop_duplicates(subset=['subtype'])
            .sort_values('subtype')
        )
        if cd.empty:
            print('No corr_df_sig rows for this (gene, tcr_col); nothing to plot.')
        else:
            gex = mdata['gex']
            tcr = mdata.obsm['tcr_embs']
            if gene not in gex.var_names:
                raise ValueError(f'{gene} not in gex.var_names')
            if j < 0 or j >= tcr.shape[1]:
                raise ValueError(f'tcr_col_idx {j} out of range')

            raw = gex[:, gene].X
            if issparse(raw):
                raw = raw.toarray().ravel()
            else:
                raw = np.asarray(raw).ravel()
            tcr_j = np.asarray(tcr[:, j]).ravel()
            sub = mdata.obs['subtype'].astype(str).to_numpy()
            cs = mdata.obs['clone_status'].astype(str).to_numpy()

            n = len(cd)
            if n == 1:
                fig, ax = plt.subplots(1, 1, figsize=(5, 4), sharex=False, sharey=False)
                axes = [ax]
            else:
                nrows = int(np.ceil(n / 2))
                fig, axes = plt.subplots(
                    nrows, 2, figsize=(10, 4 * nrows), sharex=False, sharey=False
                )
                axes = np.ravel(axes)
                for k in range(n, len(axes)):
                    axes[k].set_visible(False)
                axes = axes[:n].tolist()

            def _corr_r(x, y):
                x = np.asarray(x, dtype=float).ravel()
                y = np.asarray(y, dtype=float).ravel()
                if corr_method == 'spearman':
                    r, p = stats.spearmanr(x, y)
                elif corr_method == 'mutual_info':
                    from sklearn.feature_selection import mutual_info_regression

                    r = float(
                        mutual_info_regression(
                            x.reshape(-1, 1), y, random_state=0
                        )[0]
                    )
                    p = np.nan
                else:
                    r, p = stats.pearsonr(x, y)
                return r, p

            for ax, (_, row) in zip(axes, cd.iterrows()):
                st = str(row['subtype'])

                m = (sub == st) & np.isin(cs, tg)
                if m.sum() < 4:
                    ax.set_title(f'{st}\n(too few cells)')
                    ax.set_xlabel(tname)
                    ax.set_ylabel(f'{gene} expression (gex)')
                    continue

                m1 = m & (cs == g1)
                m2 = m & (cs == g2)

                r1 = np.nan
                r2 = np.nan
                if m1.sum() >= 2:
                    x1_tmp = tcr_j[m1]
                    y1_tmp = raw[m1]
                    if np.all(np.isfinite(x1_tmp)) and np.all(np.isfinite(y1_tmp)):
                        r1, _ = _corr_r(x1_tmp, y1_tmp)
                if m2.sum() >= 2:
                    x2_tmp = tcr_j[m2]
                    y2_tmp = raw[m2]
                    if np.all(np.isfinite(x2_tmp)) and np.all(np.isfinite(y2_tmp)):
                        r2, _ = _corr_r(x2_tmp, y2_tmp)

                ax.scatter(
                    tcr_j[m1],
                    raw[m1],
                    s=scatter_size,
                    alpha=0.35,
                    rasterized=True,
                    c='#1f77b4',
                    edgecolors='none',
                    label=g1,
                )
                ax.scatter(
                    tcr_j[m2],
                    raw[m2],
                    s=scatter_size,
                    alpha=0.35,
                    rasterized=True,
                    c='#ff7f0e',
                    edgecolors='none',
                    label=g2,
                )

                if (
                    m1.sum() >= 2
                    and np.all(np.isfinite(tcr_j[m1]))
                    and np.all(np.isfinite(raw[m1]))
                ):
                    x1 = tcr_j[m1]
                    y1 = raw[m1]
                    lr1 = stats.linregress(x1, y1)
                    xx1 = np.linspace(np.nanmin(x1), np.nanmax(x1), 100)
                    yy1 = lr1.slope * xx1 + lr1.intercept
                    r1s = f'{r1:.3f}' if np.isfinite(r1) else 'NA'
                    ax.plot(
                        xx1,
                        yy1,
                        '--',
                        c='#1f77b4',
                        lw=1.2,
                        zorder=5,
                        label=f'{g1} fit ({corr_method} r={r1s})',
                    )
                if (
                    m2.sum() >= 2
                    and np.all(np.isfinite(tcr_j[m2]))
                    and np.all(np.isfinite(raw[m2]))
                ):
                    x2 = tcr_j[m2]
                    y2 = raw[m2]
                    lr2 = stats.linregress(x2, y2)
                    xx2 = np.linspace(np.nanmin(x2), np.nanmax(x2), 100)
                    yy2 = lr2.slope * xx2 + lr2.intercept
                    r2s = f'{r2:.3f}' if np.isfinite(r2) else 'NA'
                    ax.plot(
                        xx2,
                        yy2,
                        '--',
                        c='#ff7f0e',
                        lw=1.2,
                        zorder=5,
                        label=f'{g2} fit ({corr_method} r={r2s})',
                    )

                ax.set_title(st)
                ax.set_xlabel(tname)
                ax.set_ylabel(f'{gene} expression (gex)')
                ax.legend(markerscale=3, frameon=False, fontsize=7)

            fig.suptitle(
                f'{gene} vs TCR (row 0; subtype_count={int(r["subtype_count"])})',
                y=1.02,
            )
            plt.tight_layout()
            plt.show()


In [ ]:
# Spearman rank plot
from scipy.sparse import issparse
from scipy import stats

row_idx = 1
scatter_size = 5

if pair_common_subtype.empty:
    print('pair_common_subtype is empty; run the pair_common cell first.')
else:
    r = pair_common_subtype.iloc[row_idx]
    gene = r['gene']
    j = int(r['tcr_col_idx'])
    tname = r['tcr_col']

    tg = target_groups if 'target_groups' in globals() else [
        'Tumor_multiSite_clone',
        'Tumor_singleSite_clone',
    ]
    if len(tg) < 2:
        raise ValueError('Need at least two target_groups for a two-color plot.')
    g1, g2 = tg[0], tg[1]

    if corr_df_sig.empty or 'tcr_col_idx' not in corr_df_sig.columns:
        print('corr_df_sig missing or has no tcr_col_idx; run the correlation cell first.')
    else:
        cd = (
            corr_df_sig[
                (corr_df_sig['gene'] == gene) & (corr_df_sig['tcr_col_idx'] == j)
            ]
            .drop_duplicates(subset=['subtype'])
            .sort_values('subtype')
        )
        if cd.empty:
            print('No corr_df_sig rows for this (gene, tcr_col); nothing to plot.')
        else:
            gex = mdata['gex']
            tcr = mdata.obsm['tcr_embs']
            if gene not in gex.var_names:
                raise ValueError(f'{gene} not in gex.var_names')
            if j < 0 or j >= tcr.shape[1]:
                raise ValueError(f'tcr_col_idx {j} out of range')

            raw = gex[:, gene].X
            if issparse(raw):
                raw = raw.toarray().ravel()
            else:
                raw = np.asarray(raw).ravel()
            tcr_j = np.asarray(tcr[:, j]).ravel()
            sub = mdata.obs['subtype'].astype(str).to_numpy()
            cs = mdata.obs['clone_status'].astype(str).to_numpy()

            n = len(cd)
            if n == 1:
                fig, ax = plt.subplots(1, 1, figsize=(5, 4), sharex=False, sharey=False)
                axes = [ax]
            else:
                nrows = int(np.ceil(n / 2))
                fig, axes = plt.subplots(
                    nrows, 2, figsize=(10, 4 * nrows), sharex=False, sharey=False
                )
                axes = np.ravel(axes)
                for k in range(n, len(axes)):
                    axes[k].set_visible(False)
                axes = axes[:n].tolist()

            def _spearman_r(x, y):
                x = np.asarray(x, dtype=float).ravel()
                y = np.asarray(y, dtype=float).ravel()
                r_sp, _ = stats.spearmanr(x, y)
                return r_sp

            for ax, (_, row) in zip(axes, cd.iterrows()):
                st = str(row['subtype'])

                m = (sub == st) & np.isin(cs, tg)
                if m.sum() < 4:
                    ax.set_title(f'{st}\n(too few cells)')
                    ax.set_xlabel(f'rank({tname})')
                    ax.set_ylabel(f'rank({gene})')
                    continue

                m1 = m & (cs == g1)
                m2 = m & (cs == g2)

                xr = np.full(tcr_j.shape, np.nan, dtype=float)
                yr = np.full(raw.shape, np.nan, dtype=float)
                xr[m] = stats.rankdata(tcr_j[m])
                yr[m] = stats.rankdata(raw[m])

                r1 = np.nan
                r2 = np.nan
                if m1.sum() >= 2:
                    x1_tmp = tcr_j[m1]
                    y1_tmp = raw[m1]
                    if np.all(np.isfinite(x1_tmp)) and np.all(np.isfinite(y1_tmp)):
                        r1 = _spearman_r(x1_tmp, y1_tmp)
                if m2.sum() >= 2:
                    x2_tmp = tcr_j[m2]
                    y2_tmp = raw[m2]
                    if np.all(np.isfinite(x2_tmp)) and np.all(np.isfinite(y2_tmp)):
                        r2 = _spearman_r(x2_tmp, y2_tmp)

                r_pool = np.nan
                if m.sum() >= 2:
                    r_pool, _ = stats.spearmanr(tcr_j[m], raw[m])

                ax.scatter(
                    xr[m1],
                    yr[m1],
                    s=scatter_size,
                    alpha=0.35,
                    rasterized=True,
                    c='#1f77b4',
                    edgecolors='none',
                    label=g1,
                )
                ax.scatter(
                    xr[m2],
                    yr[m2],
                    s=scatter_size,
                    alpha=0.35,
                    rasterized=True,
                    c='#ff7f0e',
                    edgecolors='none',
                    label=g2,
                )

                if (
                    m1.sum() >= 2
                    and np.all(np.isfinite(xr[m1]))
                    and np.all(np.isfinite(yr[m1]))
                ):
                    x1 = xr[m1]
                    y1 = yr[m1]
                    lr1 = stats.linregress(x1, y1)
                    xx1 = np.linspace(np.nanmin(x1), np.nanmax(x1), 100)
                    yy1 = lr1.slope * xx1 + lr1.intercept
                    r1s = f'{r1:.3f}' if np.isfinite(r1) else 'NA'
                    ax.plot(
                        xx1,
                        yy1,
                        '--',
                        c='#1f77b4',
                        lw=1.2,
                        zorder=5,
                        label=f'{g1} Spearman r={r1s}',
                    )
                if (
                    m2.sum() >= 2
                    and np.all(np.isfinite(xr[m2]))
                    and np.all(np.isfinite(yr[m2]))
                ):
                    x2 = xr[m2]
                    y2 = yr[m2]
                    lr2 = stats.linregress(x2, y2)
                    xx2 = np.linspace(np.nanmin(x2), np.nanmax(x2), 100)
                    yy2 = lr2.slope * xx2 + lr2.intercept
                    r2s = f'{r2:.3f}' if np.isfinite(r2) else 'NA'
                    ax.plot(
                        xx2,
                        yy2,
                        '--',
                        c='#ff7f0e',
                        lw=1.2,
                        zorder=5,
                        label=f'{g2} Spearman r={r2s}',
                    )

                rps = f'{r_pool:.3f}' if np.isfinite(r_pool) else 'NA'
                ax.set_title(f'{st}\n(pooled Spearman r={rps})')
                ax.set_xlabel(f'rank({tname})')
                ax.set_ylabel(f'rank({gene})')
                ax.legend(markerscale=3, frameon=False, fontsize=7)

            fig.suptitle(
                f'Rank scatter (Spearman): {gene} vs TCR (pair_common row {row_idx}; subtype_count={int(r["subtype_count"])})',
                y=1.02,
            )
            plt.tight_layout()
            plt.show()


## Hurdle model

## GSEA

In [ ]:
aa

In [ ]:
pair_common_subtype

In [ ]:
import gseapy as gp

gene_list = pair_common_subtype['gene'].unique().tolist()
print(f"Number of unique genes for GSEA: {len(gene_list)}")
print("Genes:", ', '.join(sorted(gene_list)))

gene_set_libs = [
    # 'GO_Biological_Process_2023',
    # 'GO_Molecular_Function_2023',
    # 'GO_Cellular_Component_2023',
    'KEGG_2021_Human',
    # 'Reactome_2022',
    'MSigDB_Hallmark_2020',
    
]

enr = gp.enrichr(
    gene_list=gene_list,
    gene_sets=gene_set_libs,
    organism='human',
    outdir=None,
    no_plot=True,
)

gsea_res = enr.results
gsea_res_sig = gsea_res[gsea_res['Adjusted P-value'] < 0.05].sort_values('Adjusted P-value')
print(f"\nSignificant terms (adj. p < 0.05): {len(gsea_res_sig)}")


In [ ]:
gsea_res_sig.to_csv('gsea_res_sig.csv')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for lib in gene_set_libs:
    df = gsea_res_sig[gsea_res_sig['Gene_set'] == lib]
    if df.empty:
        continue
    top = df.head(10).copy()
    top['-log10(adj p)'] = -np.log10(top['Adjusted P-value'].clip(lower=1e-300))
    top['Overlap_frac'] = top['Overlap'].apply(
        lambda x: int(x.split('/')[0]) / int(x.split('/')[1])
    )

    fig, ax = plt.subplots(figsize=(8, max(3, len(top) * 0.35)))
    bars = ax.barh(
        range(len(top)),
        top['-log10(adj p)'].values,
        color=plt.cm.RdYlBu_r(top['Overlap_frac'].values / max(top['Overlap_frac'].values.max(), 1e-9)),
        edgecolor='grey', linewidth=0.5,
    )
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top['Term'].values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('-log10(Adjusted P-value)')
    ax.set_title(f'{lib}  (top {len(top)} terms)')

    sm = plt.cm.ScalarMappable(
        cmap=plt.cm.RdYlBu_r,
        norm=plt.Normalize(0, top['Overlap_frac'].max()),
    )
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label('Overlap fraction')

    plt.tight_layout()
    plt.show()

## TCR corr with MANA score

In [ ]:
mdata_tumor = mdata[mdata.obs["source"] == "Tumor"]
mdata_tumor

In [ ]:
MANASCORE_DIR = r"E:\Biology\References\TCR and GEX\Ciculating T in tumor to predict response\Code\MANAscore"
models_dir = os.path.join(MANASCORE_DIR, "models")
os.makedirs(models_dir, exist_ok=True)

BASE_URL = "https://media.githubusercontent.com/media/BKI-immuno-KNS/MANAscore/main/MANAscore/models"
model_files = ["voting_i_classifier.pkl.gz", "voting_ni_classifier.pkl.gz"]

for fname in model_files:
    fpath = os.path.join(models_dir, fname)
    if not os.path.exists(fpath):
        url = f"{BASE_URL}/{fname}"
        print(f"Downloading {fname} from GitHub...")
        urllib.request.urlretrieve(url, fpath)
        print(f"  Saved to {fpath}")
    else:
        print(f"{fname} already exists locally.")

sys.path.insert(0, os.path.dirname(MANASCORE_DIR))
from MANAscore import MANAscore

scorer = MANAscore()
scorer.load_voting_models()
print("\nMANAscore models loaded successfully.")

In [ ]:
MANA_GENES = ["CXCL13", "ENTPD1", "IL7R"]
adata_gex = mdata_tumor["gex"]

missing = [g for g in MANA_GENES if g not in adata_gex.var_names]

if missing:
    print(f"Genes {missing} not in HVG subset ({adata_gex.n_vars} genes). Loading from full dataset...")
    full_mdata = mu.read("GSE139555_Tcells.h5mu")
    full_gex = full_mdata["gex"]
    cells = adata_gex.obs_names
    X_3g = full_gex[cells, MANA_GENES].X
    if hasattr(X_3g, "toarray"):
        X_3g = X_3g.toarray()
    gene_expr = pd.DataFrame(X_3g, index=cells, columns=MANA_GENES)
    del full_mdata, full_gex
else:
    print(f"All 3 MANAscore genes found in GEX modality.")
    X_3g = adata_gex[:, MANA_GENES].X
    if hasattr(X_3g, "toarray"):
        X_3g = X_3g.toarray()
    gene_expr = pd.DataFrame(X_3g, index=adata_gex.obs_names, columns=MANA_GENES)

gene_expr.describe()

In [ ]:
# Non-imputed classifier (standard scRNA-seq without MAGIC/similar imputation)
prob_ni = scorer.voting_ni_classifier.predict_proba(gene_expr[MANA_GENES])[:, 1]
mdata_tumor["gex"].obs["MANAscore"] = prob_ni

# Imputed classifier (use if data was imputed with MAGIC etc.)
prob_i = scorer.voting_i_classifier.predict_proba(gene_expr[MANA_GENES])[:, 1]
mdata_tumor["gex"].obs["MANAscore_imputed"] = prob_i

mdata_tumor.update()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sc.pl.violin(mdata_tumor["gex"], keys="MANAscore", groupby="subtype",
             rotation=45, ax=axes[0], show=False)
axes[0].set_title("MANAscore by Subtype")

sc.pl.violin(mdata_tumor["gex"], keys="MANAscore", groupby="source",
             rotation=45, ax=axes[1], show=False)
axes[1].set_title("MANAscore by Source (Tumor vs Blood)")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr, pearsonr
from statsmodels.stats.multitest import multipletests

tcr_X = np.asarray(mdata_tumor.obsm['tcr_embs'])
feat_names = list(mdata_tumor.uns.get('tcr_embs_feature_names', []))
if len(feat_names) != tcr_X.shape[1]:
    feat_names = [f'tcr_emb_{i}' for i in range(tcr_X.shape[1])]

mana = mdata_tumor["gex"].obs["MANAscore"].values
valid = ~np.isnan(mana)
mana = mana[valid]
tcr_X = tcr_X[valid]
print(f"Computing correlations: {tcr_X.shape[1]} TCR features x {len(mana)} tumor cells")

records = []
for j in range(tcr_X.shape[1]):
    col = tcr_X[:, j]
    if np.std(col) == 0:
        continue
    r_s, p_s = spearmanr(mana, col)
    r_p, p_p = pearsonr(mana, col)
    records.append({
        'feature': feat_names[j],
        'spearman_r': r_s, 'spearman_p': p_s,
        'pearson_r': r_p, 'pearson_p': p_p,
    })

corr_df = pd.DataFrame(records)
corr_df['spearman_p_adj'] = multipletests(corr_df['spearman_p'], method='fdr_bh')[1]
corr_df['pearson_p_adj'] = multipletests(corr_df['pearson_p'], method='fdr_bh')[1]
corr_df['abs_spearman'] = corr_df['spearman_r'].abs()
corr_df = corr_df.sort_values('abs_spearman', ascending=False).reset_index(drop=True)

n_sig = (corr_df['pearson_p_adj'] < 0.05).sum()
print(f"Significant (FDR < 0.05, Pearson): {n_sig} / {len(corr_df)}")
print(f"\nTop 20 TCR features correlated with MANAscore:")
corr_df.head(20)

In [ ]:
top_n = min(30, len(corr_df))
top = corr_df.head(top_n).iloc[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, top_n * 0.28)))

colors_s = ['#d62728' if r > 0 else '#1f77b4' for r in top['spearman_r']]
axes[0].barh(range(top_n), top['spearman_r'].values, color=colors_s)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top['feature'].values, fontsize=8)
for i, (r, p) in enumerate(zip(top['spearman_r'], top['spearman_p_adj'])):
    marker = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    axes[0].text(r, i, f' {marker}', va='center', fontsize=7)
axes[0].set_xlabel('Spearman r')
axes[0].set_title(f'Top {top_n} TCR features vs MANAscore (Spearman)')
axes[0].axvline(0, color='grey', lw=0.5)

colors_p = ['#d62728' if r > 0 else '#1f77b4' for r in top['pearson_r']]
axes[1].barh(range(top_n), top['pearson_r'].values, color=colors_p)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(top['feature'].values, fontsize=8)
for i, (r, p) in enumerate(zip(top['pearson_r'], top['pearson_p_adj'])):
    marker = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    axes[1].text(r, i, f' {marker}', va='center', fontsize=7)
axes[1].set_xlabel('Pearson r')
axes[1].set_title(f'Top {top_n} TCR features vs MANAscore (Pearson)')
axes[1].axvline(0, color='grey', lw=0.5)

plt.tight_layout()
plt.show()

sig_df = corr_df[corr_df['spearman_p_adj'] < 0.05].copy()
print(f"\nAll {len(sig_df)} significant features (FDR < 0.05):")
print(sig_df[['feature','spearman_r','spearman_p_adj','pearson_r','pearson_p_adj']].to_string(index=False))

In [ ]:
top_k = 9
top_scatter = corr_df.head(top_k)

ncols = 3
nrows = int(np.ceil(top_k / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

feat_idx_map = {fn: j for j, fn in enumerate(feat_names)}

for i, row in enumerate(top_scatter.itertuples()):
    ax = axes[i]
    j = feat_idx_map[row.feature]
    x = tcr_X[:, j]
    ax.scatter(x, mana, s=3, alpha=0.3, rasterized=True)
    ax.set_xlabel(row.feature, fontsize=9)
    ax.set_ylabel('MANAscore', fontsize=9)

    z = np.polyfit(x, mana, 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, np.polyval(z, xline), color='red', lw=1.5)

    sig = '***' if row.spearman_p_adj < 0.001 else '**' if row.spearman_p_adj < 0.01 else '*' if row.spearman_p_adj < 0.05 else 'ns'
    ax.set_title(f"ρ={row.spearman_r:.3f} {sig}", fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'Top {top_k} TCR features vs MANAscore (by |Spearman r|)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Save data as csv

In [ ]:
mdata_tumor

In [ ]:
from scipy.sparse import issparse

mdata = mdata_tumor

tcr_feature_names = list(mdata.uns.get('tcr_embs_feature_names', []))
tcr_all = np.asarray(mdata.obsm['tcr_embs'])
if len(tcr_feature_names) != tcr_all.shape[1]:
    tcr_feature_names = [f'tcr_emb_{i}' for i in range(tcr_all.shape[1])]

gene_names = mdata['gex'].var_names.tolist()
gex_mat = mdata['gex'].X

X_gex = gex_mat
if issparse(X_gex):
    X_gex = X_gex.toarray()

df_gex = pd.DataFrame(X_gex, columns=gene_names, index=mdata.obs_names)
df_tcr = pd.DataFrame(tcr_all, columns=tcr_feature_names, index=mdata.obs_names)

meta = pd.DataFrame({
    'subtype': mdata.obs['subtype'].values,
    'clone_status': mdata.obs['clone_status'].values,
    'clone_id_size': mdata['airr'].obs['clone_id_size'].values,
    'MANAscore': mdata['gex'].obs['MANAscore'].values,
}, index=mdata.obs_names)

df_all = pd.concat([meta, df_gex, df_tcr], axis=1)
fname_all = os.path.join(outdir, 'gex_tcr_all.csv')
df_all.to_csv(fname_all)
print(f"Saved {fname_all}  ({df_all.shape[0]} cells x {df_all.shape[1]} cols)")

df_gex_out = pd.concat([meta, df_gex], axis=1)
fname_gex = os.path.join(outdir, 'gex_all.csv')
df_gex_out.to_csv(fname_gex)
print(f"Saved {fname_gex}  ({df_gex_out.shape[0]} cells x {df_gex_out.shape[1]} cols)")

df_tcr_out = pd.concat([meta, df_tcr], axis=1)
fname_tcr = os.path.join(outdir, 'tcr_emb_all.csv')
df_tcr_out.to_csv(fname_tcr)
print(f"Saved {fname_tcr}  ({df_tcr_out.shape[0]} cells x {df_tcr_out.shape[1]} cols)")

In [ ]:
aa

## feature weights by classification  (No correlations)

In [ ]:
## select one cell from each clonotype
airr_obs = mdata['airr'].obs.copy()
clone_id_col = airr_obs['clone_id']
if hasattr(clone_id_col, 'cat'):
    clone_id_col = clone_id_col.astype(str)
airr_obs['_clone_id_str'] = clone_id_col

expanded = airr_obs[airr_obs['clone_id_size'] > 1].dropna(subset=['_clone_id_str'])
sampled_expanded_idx = (
    expanded
    .groupby('_clone_id_str', observed=True)
    .sample(n=1, random_state=42)
    .index
)

single_idx = airr_obs[airr_obs['clone_id_size'] == 1].index
keep_idx = sampled_expanded_idx.append(single_idx)

mdata_sub = mdata[keep_idx].copy()
print(f"Expanded clones: {len(expanded)} cells -> {len(sampled_expanded_idx)} (1 per clone)")
print(f"Single clones: {len(single_idx)}")
print(f"Total after dedup: {mdata_sub.n_obs} cells")

mdata_sub = mdata_sub[~mdata_sub.obs['source'].isin(['Blood'])]
mdata_sub.obs['source'].value_counts()

In [ ]:
# Combine 'NAT_multiSite_clone' and 'Tumor_multiSite_clone' into 'multisite'
# mdata_sub = mdata[mdata.obs['cloned'].astype(str) == 'True'].copy()

clone_status = mdata_sub.obs['clone_status'].copy()
clone_status = clone_status.replace({'NAT_multiSite_clone': 'multisite', 
            'Tumor_multiSite_clone': 'multisite'})

mdata_sub.obs['TCR_clone_type'] = clone_status
mdata_sub.obs['TCR_clone_type'] = mdata_sub.obs['TCR_clone_type'].cat.remove_unused_categories()


In [ ]:
mdata_sub.obs['subtype'].value_counts()

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# mdata_sub = mdata_sub[mdata_sub.obs['subtype'].astype('str') == 'CD8_Teff']

# --- features & targets (drop clone_id_size) ---
X_all = np.asarray(mdata_sub.obsm['tcr_embs'])
feature_names_all = list(mdata_sub.uns.get('tcr_embs_feature_names', []))
if len(feature_names_all) != X_all.shape[1]:
    feature_names_all = [f'tcr_emb_{i}' for i in range(X_all.shape[1])]

drop_features = {'clone_id_size'}
keep_mask = np.array([fn not in drop_features for fn in feature_names_all])
X = X_all[:, keep_mask]
feature_names = [fn for fn in feature_names_all if fn not in drop_features]
print(f"Dropped {(~keep_mask).sum()} feature(s): {drop_features & set(feature_names_all)}")

# set the target label here ####
target_obs = mdata_sub.obs['TCR_clone_type'].cat.remove_unused_categories()
y_raw = target_obs.astype(str).values
le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_
print(f"Features: {X.shape[1]},  Samples: {X.shape[0]},  Classes: {list(class_names)}")


# --- XGBoost with 5-fold stratified CV ---
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred = cross_val_predict(xgb, X, y, cv=cv)

print("\n=== XGBoost 5-Fold CV Classification Report ===")
print(classification_report(y, y_pred, target_names=class_names))

# --- Refit on full data to get feature importances ---
xgb.fit(X, y)
importances = xgb.feature_importances_
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)

# --- Confusion matrix ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ConfusionMatrixDisplay.from_predictions(
    le.inverse_transform(y), le.inverse_transform(y_pred),
    ax=axes[0], cmap='Blues', xticks_rotation=45,
)
axes[0].set_title('XGBoost 5-Fold CV  Confusion Matrix')

# --- Top 30 feature importances ---
top_n = min(30, len(imp_df))
top = imp_df.head(top_n).iloc[::-1]
axes[1].barh(range(top_n), top['importance'].values, color='steelblue')
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(top['feature'].values)
axes[1].set_xlabel('Feature importance (gain)')
axes[1].set_title(f'Top {top_n} TCR embedding features (XGBoost)')

plt.tight_layout()
plt.show()

print(f"\nTop 10 features:\n{imp_df.head(10).to_string(index=False)}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

y_pred_rf = cross_val_predict(rf, X, y, cv=cv)

print("=== Random Forest 5-Fold CV Classification Report ===")
print(classification_report(y, y_pred_rf, target_names=class_names))

rf.fit(X, y)
imp_rf = pd.DataFrame({'feature': feature_names, 'importance': rf.feature_importances_})
imp_rf = imp_rf.sort_values('importance', ascending=False).reset_index(drop=True)

# --- Side-by-side: RF confusion matrix + RF top features ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ConfusionMatrixDisplay.from_predictions(
    le.inverse_transform(y), le.inverse_transform(y_pred_rf),
    ax=axes[0], cmap='Oranges', xticks_rotation=45,
)
axes[0].set_title('Random Forest 5-Fold CV  Confusion Matrix')

top_n = min(30, len(imp_rf))
top_rf = imp_rf.head(top_n).iloc[::-1]
axes[1].barh(range(top_n), top_rf['importance'].values, color='coral')
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(top_rf['feature'].values)
axes[1].set_xlabel('Feature importance (Gini / MDI)')
axes[1].set_title(f'Top {top_n} TCR embedding features (Random Forest)')

plt.tight_layout()
plt.show()

# --- Compare top features from both models ---
comp = imp_df[['feature', 'importance']].head(20).merge(
    imp_rf[['feature', 'importance']].head(20),
    on='feature', how='outer', suffixes=('_xgb', '_rf')
).fillna(0).sort_values('importance_xgb', ascending=False)
print("\nTop 20 features comparison (XGBoost vs RandomForest):")
print(comp.to_string(index=False))

In [ ]:
len(pair_common_subtype['tcr_col'].unique().tolist())

In [ ]:
# Use TCR features from pair_common_subtype (DEG-correlated features shared across subtypes)
top_feats = pair_common_subtype['tcr_col'].unique().tolist()
top_feats = [f for f in top_feats if f in feature_names]
top_k = len(20)
feat_idx = [feature_names.index(f) for f in top_feats]
X_top = X[:, feat_idx]
print(f"Retraining with {top_k} features from pair_common_subtype: {top_feats}")

# --- XGBoost on top-k features ---
xgb_top = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=42, n_jobs=-1,
)
y_pred_top = cross_val_predict(xgb_top, X_top, y, cv=cv)

print(f"\n=== XGBoost 5-Fold CV  (top {top_k} features) ===")
print(classification_report(y, y_pred_top, target_names=class_names))

xgb_top.fit(X_top, y)
imp_top = pd.DataFrame({'feature': top_feats, 'importance': xgb_top.feature_importances_})
imp_top = imp_top.sort_values('importance', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ConfusionMatrixDisplay.from_predictions(
    le.inverse_transform(y), le.inverse_transform(y_pred_top),
    ax=axes[0], cmap='Purples', xticks_rotation=45,
)
axes[0].set_title(f'XGBoost CV  top {top_k} features  Confusion Matrix')

top_plot = imp_top.iloc[::-1]
axes[1].barh(range(top_k), top_plot['importance'].values, color='mediumpurple')
axes[1].set_yticks(range(top_k))
axes[1].set_yticklabels(top_plot['feature'].values)
axes[1].set_xlabel('Feature importance (gain)')
axes[1].set_title(f'Feature importance (top {top_k} retrained)')

plt.tight_layout()
plt.show()

from sklearn.metrics import accuracy_score, f1_score
acc_all = accuracy_score(y, y_pred)
acc_top = accuracy_score(y, y_pred_top)
f1_all = f1_score(y, y_pred, average='weighted')
f1_top = f1_score(y, y_pred_top, average='weighted')
print(f"\nAll features ({X.shape[1]}):  Accuracy={acc_all:.4f}  Weighted-F1={f1_all:.4f}")
print(f"Top {top_k} features:       Accuracy={acc_top:.4f}  Weighted-F1={f1_top:.4f}")
print(f"Delta:                Accuracy={acc_top-acc_all:+.4f}  Weighted-F1={f1_top-f1_all:+.4f}")

## TCR features separation  (Not useful)

In [ ]:
aa

In [ ]:
mdata_sub.obs['cloned'].value_counts()

In [ ]:
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

subtypes = mdata_sub.obs['subtype'].cat.remove_unused_categories().astype(str).unique().tolist()
for st in subtypes:
    m_cloned_subtypes = mdata_sub[(mdata_sub.obs['subtype'].astype(str) == st)].copy()

    tcr_embs = np.asarray(m_cloned_subtypes.obsm['tcr_embs'])
    source = m_cloned_subtypes.obs['clone_status'].values
    feature_names = list(m_cloned_subtypes.uns.get('tcr_embs_feature_names', []))

    if not feature_names:
        feature_names = [f'tcr_emb_{i}' for i in range(tcr_embs.shape[1])]

    groups = {s: tcr_embs[source == s] for s in np.unique(source)}

    records = []
    for j in range(tcr_embs.shape[1]):
        col_groups = [g[:, j] for g in groups.values()]
        col_groups = [g[np.isfinite(g)] for g in col_groups]
        if any(len(g) < 2 for g in col_groups):
            continue
        h_stat, p_val = kruskal(*col_groups)
        K = len(col_groups)
        N = sum(len(g) for g in col_groups)
        epsilon_sq = (h_stat - K + 1) / (N - K)

        records.append({
            'feature_idx': j,
            'feature': feature_names[j] if j < len(feature_names) else f'tcr_emb_{j}',
            'H_statistic': h_stat,
            'p_value': p_val,
            'epsilon_sq':epsilon_sq
        })

    kw_df = pd.DataFrame(records)
    kw_df.dropna(subset=['p_value'], inplace=True)

    _, kw_df['p_adj'], _, _ = multipletests(kw_df['p_value'], method='fdr_bh')
    kw_df = kw_df.sort_values('p_adj', ascending=True).reset_index(drop=True)
    kw_df['rank'] = kw_df.index + 1

    print(f"Kruskal-Wallis test on {st} ")
    print(kw_df.head(8))
    print(f"({', '.join(f'{k}: n={v.shape[0]}' for k, v in groups.items())})")
    print(f"Significant features (FDR < 0.05): {(kw_df['p_adj'] < 0.1).sum()} / {len(kw_df)}\n")


In [ ]:
kw_df.head(8)

In [ ]:
# Power analysis
from scipy.stats import chi2, ncx2

def kruskal_power(epsilon_sq, N, K, alpha=0.05, n_sim=1000):
    """
    Estimate power given observed effect size and proposed N.
    Uses noncentral chi2 approximation.
    """
    df = K - 1
    # Noncentrality parameter
    lambda_ = N * epsilon_sq * (K) / (1 - epsilon_sq)
    critical_val = chi2.ppf(1 - alpha, df)
    power = 1 - ncx2.cdf(critical_val, df, lambda_)
    return power

N = len(mdata_sub)
# Example: what N do I need for ε²=0.06 to reach 80% power?
for n in [N,N*2]:
    pw = kruskal_power(epsilon_sq=0.06, N=n, K=4)
    print(f"N={n:5d}: power={pw:.3f}")

In [ ]:
top_n_plot = min(10, len(kw_df))
top = kw_df.iloc[1:top_n_plot+1,:].copy()

fig, ax = plt.subplots(figsize=(8, max(4, top_n_plot * 0.28)))
colors = ['#c44e52' if p < 0.05 else '#8c8c8c' for p in top['p_adj']]
ax.barh(range(top_n_plot), top['H_statistic'].values, color=colors)
ax.set_yticks(range(top_n_plot))
ax.set_yticklabels(top['feature'].values, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Kruskal-Wallis H statistic')
ax.set_title(f"Top {top_n_plot} TCR embedding dims ranked by source separation\n"
             f"(red = FDR < 0.05, grey = n.s.)")
plt.tight_layout()
plt.show()

## Trajectory  (useless)

In [ ]:
aa

In [ ]:
# adata_tumor = mdata[mdata.obs['source'] == 'Tumor'].copy()
adata = mdata['gex']


In [ ]:
sc.pp.neighbors(adata, n_neighbors=60, use_rep='X_pca')
sc.tl.diffmap(adata, n_comps=30)
sc.pp.neighbors(adata, n_neighbors=20, use_rep="X_diffmap")
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata)

In [ ]:
sc.tl.draw_graph(adata)

In [ ]:
sc.pl.draw_graph(adata, color=["subtype", "source"], legend_loc="on data")

In [ ]:
sc.tl.leiden(adata)
sc.tl.paga(adata, groups="leiden")

In [ ]:
sc.pl.paga(adata)